In [4]:
#THIS FILE ONLY PREPS THE DATA FOR PRELIMINARY RESEARCH ON WEKA (e.g creation of weak labels, saving as csv, etc.)

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd



def build_range_weak_labels(features: pd.DataFrame,
                            *,
                            label_window: int = 100,
                            range_threshold: float = 0.432444,
                            not_range_threshold: float = 0.294024) -> pd.Series:

    
    """
    Creates weak labels for range behavior.

    Label meaning:
        "range" = range-like
        "not_range" = not range-like
        NaN = unclear / ignored

    Important:
        This is NOT a future-return label.
        This is a descriptive current-window behavior label.
    """

    candidate_col = f"range_behavior_candidate_{label_window}"

    if candidate_col not in features.columns:
        raise ValueError(
            f"Missing label source column: {candidate_col}. "
            "Make sure your saved feature dataframe includes this column."
        )

    candidate = features[candidate_col]

    labels = pd.Series(np.nan, index=features.index, name="range_label", dtype="object")

    labels[candidate >= range_threshold] = "range"
    labels[candidate <= not_range_threshold] = "not_range"

    return labels


def time_ordered_split(df: pd.DataFrame,
                       *,
                       train_size: float = 0.80) -> tuple[pd.DataFrame, pd.DataFrame]:
    
    """
    Splits dataframe in chronological/original order.

    Do not shuffle market data.
    """

    split_idx = int(len(df) * train_size)

    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()

    return train_df, test_df


def make_weka_range_dataset(feature_path: str | Path,
                            *,
                            output_dir: str | Path = "feature_data/weka",
                            symbol: str = "EURUSD",
                            timeframe: str = "D1",
                            label_window: int = 100,
                            range_threshold: float = 0.70,
                            not_range_threshold: float = 0.40,
                            train_size: float = 0.80,
                            overwrite: bool = False) -> pd.DataFrame:
    
    """
    Converts saved range training features into WEKA-ready CSV files.

    Saves:
        1. Full labeled WEKA dataset
        2. Chronological train dataset
        3. Chronological test dataset

    The class column is range_label and is placed last.
    """

    feature_path = Path(feature_path)
    output_dir = Path(output_dir)

    if not feature_path.exists():
        raise FileNotFoundError(f"Feature file does not exist: {feature_path}")

    output_dir.mkdir(parents=True, exist_ok=True)

    full_output_path = output_dir / f"{symbol}_{timeframe}_range_weka_full.csv"
    train_output_path = output_dir / f"{symbol}_{timeframe}_range_weka_train.csv"
    test_output_path = output_dir / f"{symbol}_{timeframe}_range_weka_test.csv"

    output_paths = [full_output_path, train_output_path, test_output_path]

    existing_files = [path for path in output_paths if path.exists()]

    if existing_files and not overwrite:
        existing_text = "\n".join(str(path) for path in existing_files)
        raise FileExistsError(
            f"WEKA file(s) already exist:\n{existing_text}\n"
            "Use overwrite=True if you want to replace them."
        )


    # load saved training-ready features

    features = pd.read_parquet(feature_path)

    # create the weak labels

    labels = build_range_weak_labels(
        features,
        label_window=label_window,
        range_threshold=range_threshold,
        not_range_threshold=not_range_threshold,
    )

    weka_df = features.copy()
    weka_df["range_label"] = labels


    # remove unclear rows

    weka_df = weka_df.dropna(subset=["range_label"]).copy()


    #clean values for WEKA

    weka_df = weka_df.replace([np.inf, -np.inf], np.nan)

    #drop columns that are completely NaN
    all_nan_cols = weka_df.columns[weka_df.isna().all()].tolist()

    if all_nan_cols:
        weka_df = weka_df.drop(columns=all_nan_cols)

    # WEKA can handle missing values as "?", but pandas CSV writes NaN as blank by default.
    # Blank missing values are usually read correctly by WEKA from CSV.
    # If needed later, we can export ARFF with explicit "?".

    # Put class column last
    feature_cols = [col for col in weka_df.columns if col != "range_label"]
    weka_df = weka_df[feature_cols + ["range_label"]]


    # time-ordered train/test split

    train_df, test_df = time_ordered_split(
        weka_df,
        train_size=train_size,
    )

    # save CSV files

    weka_df.to_csv(full_output_path, index=False)
    train_df.to_csv(train_output_path, index=False)
    test_df.to_csv(test_output_path, index=False)


    print("[WEKA EXPORT] Saved full WEKA dataset to:", full_output_path)
    print("[WEKA EXPORT] Saved WEKA train dataset to:", train_output_path)
    print("[WEKA EXPORT] Saved WEKA test dataset to:", test_output_path)
    print()
    print("[WEKA EXPORT] Full shape:", weka_df.shape)
    print("[WEKA EXPORT] Train shape:", train_df.shape)
    print("[WEKA EXPORT] Test shape:", test_df.shape)
    print()
    print("[WEKA EXPORT] Dropped all-NaN columns:", all_nan_cols)
    print()
    print("[WEKA EXPORT] Label distribution:")
    print(weka_df["range_label"].value_counts(normalize=True))
    print()
    print("[WEKA EXPORT] Label counts:")
    print(weka_df["range_label"].value_counts())

    print( features["range_behavior_candidate_100"].describe() )

    print( features["range_behavior_candidate_100"].quantile([0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]) )
    
    return weka_df


if __name__ == "__main__":

    feature_path = "feature_data/range_training_features/EURUSD_D1_range_training_features.parquet"

    make_weka_range_dataset(
        feature_path,
        output_dir="feature_data/weka",
        symbol="EURUSD",
        timeframe="D1",
        label_window=100,
        range_threshold=0.432444,
        not_range_threshold=0.294024,
        train_size=0.80,
        overwrite=True
    )

[WEKA EXPORT] Saved full WEKA dataset to: feature_data/weka/EURUSD_D1_range_weka_full.csv
[WEKA EXPORT] Saved WEKA train dataset to: feature_data/weka/EURUSD_D1_range_weka_train.csv
[WEKA EXPORT] Saved WEKA test dataset to: feature_data/weka/EURUSD_D1_range_weka_test.csv

[WEKA EXPORT] Full shape: (578, 1483)
[WEKA EXPORT] Train shape: (462, 1483)
[WEKA EXPORT] Test shape: (116, 1483)

[WEKA EXPORT] Dropped all-NaN columns: ['touch_balance_ratio_20_50_z250', 'two_sided_touch_score_ratio_20_50_z250', 'touch_balance_ratio_50_100_z250', 'two_sided_touch_score_ratio_50_100_z250']

[WEKA EXPORT] Label distribution:
range_label
not_range    0.5
range        0.5
Name: proportion, dtype: float64

[WEKA EXPORT] Label counts:
range_label
not_range    289
range        289
Name: count, dtype: int64
count    2885.000000
mean        0.354925
std         0.058073
min         0.237129
25%         0.315779
50%         0.342774
75%         0.377479
max         0.581262
Name: range_behavior_candidate_100